## VideoFake | Face Swap / Deep Fake Colab (English edition without NSFW filter by NeuroModern)
Based on [Facefusion](https://github.com/facefusion/), [roop by s0md3v](https://github.com/s0md3v/roop) and [dream80 colab](https://github.com/dream80/roop_colab/)

**Updated for current Google Colab** (Python 3.12+/3.13, CUDA 12/13). Runtime type: **GPU**.

Please follow me on the twitter: [twitter.com/neuromodern](https://twitter.com/neuromodern)
<a href="https://colab.research.google.com/github/ixelizm/VideoFake/blob/main/VideoFake_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title 0. Check GPU / CUDA (do NOT install nvidia-cudnn — Colab already has it)
import sys
import subprocess

print("Python:", sys.version)
subprocess.run(["nvidia-smi"], check=False)

try:
    import torch
    print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available(), "| CUDA:", torch.version.cuda)
except Exception as e:
    print("torch check skipped:", e)

!ffmpeg -version | head -n 1

In [ ]:
#@title 1. Install repository
import os

REPO_DIR = "/content/VideoFake"
if os.path.isdir(os.path.join(REPO_DIR, "roop")):
    print(f"Repo already present at {REPO_DIR} — pulling latest")
    %cd /content/VideoFake
    !git pull --ff-only || git fetch --depth 1 origin main && git reset --hard origin/main
else:
    !git clone --depth 1 https://github.com/ixelizm/VideoFake.git {REPO_DIR}

In [ ]:
#@title 2. Create model folder and set dir
import os

directory = "/content/VideoFake/roop/models/"
os.makedirs(directory, exist_ok=True)
print(f"Models directory: {directory}")
%cd /content/VideoFake/roop

In [ ]:
#@title 3. Install requirements (Colab-compatible)
# basicsr/gfpgan break on Python 3.13 via pip; install_colab.py patches and installs them.
%cd /content/VideoFake/roop
!python install_colab.py

In [ ]:
#@title 4. Set up video and face sources
source = "/content/VideoFake/face.jpg" #@param {type:"string"}
target = "/content/VideoFake/input_video.mp4" #@param {type:"string"}
output = "/content/VideoFake/out.mp4" #@param {type:"string"}

import os
for label, path in (("source", source), ("target", target)):
    print(f"{label}: {path} | exists={os.path.isfile(path)}")

In [ ]:
#@title 5. Run
import os, sys, subprocess, shlex

# Defaults if cell 4 was skipped
source = globals().get("source", "/content/VideoFake/face.jpg")
target = globals().get("target", "/content/VideoFake/input_video.mp4")
output = globals().get("output", "/content/VideoFake/out.mp4")

Device = "cuda" #@param ["cuda", "cpu"]
Processor = "face_swapper" #@param ["face_swapper face_enhancer", "face_swapper", "face_enhancer"]
VideoEncoder = "libx264" #@param ["libx264", "libx265", "libvpx-vp9"]
VideoQuality = "18" #@param {type:"string"}
KeepFPS = True #@param {type:"boolean"}
KeepAudio = True #@param {type:"boolean"}
KeepFrames = False #@param {type:"boolean"}
ManyFaces = True #@param {type:"boolean"}

print("source:", source, "exists:", os.path.isfile(source))
print("target:", target, "exists:", os.path.isfile(target))
print("output:", output)
assert os.path.isfile(source), "source image missing — upload face.jpg or fix path in cell 4"
assert os.path.isfile(target), "target video missing — upload input_video.mp4 or fix path in cell 4"
assert os.path.isdir("/content/VideoFake/roop"), "repo missing — run cell 1 (clone)"

try:
    import onnxruntime as ort
    print("onnxruntime", ort.__version__, "providers", ort.get_available_providers())
    if Device == "cuda" and "CUDAExecutionProvider" not in ort.get_available_providers():
        print("CUDA provider missing — using cpu")
        Device = "cpu"
except Exception as e:
    raise RuntimeError("onnxruntime not installed — re-run cell 3") from e

args = [
    sys.executable, "run.py",
    "--execution-provider", Device,
    "--source", source,
    "-t", target,
    "-o", output,
    "--frame-processor", *Processor.split(),
    "--output-video-encoder", VideoEncoder,
    "--output-video-quality", str(VideoQuality),
]
if KeepFPS: args.append("--keep-fps")
if not KeepAudio: args.append("--skip-audio")
if KeepFrames: args.append("--keep-frames")
if ManyFaces: args.append("--many-faces")

print("cmd:", " ".join(shlex.quote(a) for a in args))
proc = subprocess.run(args, cwd="/content/VideoFake/roop")
print("exit code:", proc.returncode)

if os.path.isfile(output) and os.path.getsize(output) > 0:
    mb = os.path.getsize(output) / (1024 * 1024)
    print(f"OUTPUT OK: {output} ({mb:.2f} MB)")
else:
    print(f"OUTPUT MISSING/EMPTY: {output}")
    print("Listing /content/VideoFake:", os.listdir("/content/VideoFake"))
    raise RuntimeError("out.mp4 was not created — scroll up for the real error in the log")

In [ ]:
#@title 6. Download result
import os
from google.colab import files

print("Looking for:", output)
if os.path.isfile(output) and os.path.getsize(output) > 0:
    print("Downloading...", round(os.path.getsize(output) / 1024 / 1024, 2), "MB")
    files.download(output)
else:
    print("File not found or empty. Re-run cell 5 and check the log for 'Processing to video succeed!'")